# Contribution recovery · run-bound evidence

**Run All reads saved, hashed artifact inputs and renders only.** It cannot generate or fit. Legacy records are manifest/input-hash compatible only; historical smoke receipts are verified under their *recorded* identities, never recertified as current fits. All scientific conclusions remain **pending review for this exact run**.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd() / '../../../..').resolve()))
from studies.recovery.reporting import read_report
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
evidence = read_report(Path.cwd())
display(Markdown(f'**Evidence ID:** `{evidence["run_id"]}` · **Source:** `{evidence["kind"]}` · '
                 f'**Provenance:** {evidence["provenance"]}'))

**Evidence ID:** `f2dccfb682b95c9e8da4ffdc0d3f8c5f4134098468d54ef4f8c8e974cfbbffb3` · **Source:** `tv_nuts` · **Provenance:** legacy manifest/input-hash compatibility only; output and source hashes unavailable

## Configuration, provenance and coverage

The evidence ID hashes the resolved config, input files, report sources and environment. Changed inputs fail Run All. Eligible channels exclude exact known-zero effects; a zero denominator means **not represented**, not zero failures. Small groups (<10 evaluable channels) are flagged.

In [2]:
display(Markdown('**Resolved configuration:**'))
display(pd.json_normalize(evidence['resolved_config']).T.rename(columns={0: 'resolved value'}))
display(pd.DataFrame([{'method': k, 'worlds': len({r['world_id'] for r in evidence['worlds'] if r.get('method', evidence['kind']) == k}), 'channels': sum(1 for c in evidence['channels'] if c.get('method', evidence['kind']) == k), 'eligible': sum(c['nonzero_effect'] for c in evidence['channels'] if c.get('method', evidence['kind']) == k)} for k in evidence['pattern_rates']]))
display(Markdown('**Optional exploration:** ' + str(evidence['capabilities'])))

**Resolved configuration:**

,resolved value
schema,1
diagnostics.bfmi,0.3
diagnostics.divergences,0
diagnostics.ess,400
diagnostics.maxdepth_hits,0
diagnostics.rhat,1.01
generator.carryover_family,geometric
generator.edge_budget.cc,"[0, 0]"
generator.edge_budget.cy,"[1, 15]"
generator.edge_budget.dc,"[0, 0]"


,method,worlds,channels,eligible
0,tv_nuts,50,799,384


**Optional exploration:** {'optional_logistic_screen': 'deferred/unsupported', 'optional_tsne': 'deferred/unsupported', 'optional_correlated_pairs': 'deferred/unsupported'}

## Noise, true parameters and variation

These are **oracle descriptive profiles**, not fit initialization and not predictors of recovery. Noise SD, baseline variation, media/control contribution SD, carryover alpha, MM multiplier and direct coefficients come from saved worlds.

In [3]:
profiles = pd.json_normalize([{'world': r['world_id'], 'method': r.get('method', evidence['kind']), **r['profile']} for r in evidence['worlds']])
display(profiles[['world', 'method', 'realized_noise_sd', 'baseline_temporal_sd', 'carryover_alpha', 'mm_kappa_mult', 'media_beta', 'control_coefficient', 'media_true_temporal_sd', 'control_true_temporal_sd']].head(60))

,world,method,realized_noise_sd,baseline_temporal_sd,carryover_alpha,mm_kappa_mult,media_beta,control_coefficient,media_true_temporal_sd,control_true_temporal_sd
0,0,tv_nuts,0.315691,0.455404,"[0.2369566410779953, 0.5683885216712952, 0.518...","[11.978698219438716, 0.07262298481904934, 3.80...","[9.800710548456635, 1.9169488226177898, 9.2557...","[-0.11812368866036463, 0.03488309766867398, -0...","[0.0, 0.26858849615731695, 0.9209901691508734,...","[0.0, 0.0, 0.04275868584238596]"
1,1,tv_nuts,0.508518,0.740878,"[0.5992437601089478, 0.5586657524108887, 0.548...","[0.13122343789213747, 19.522093630987555, 18.0...","[1.2602244107891194, 27.107757575937807, 24.21...","[0.11815139276876277, 0.14041838546046895, -0....","[0.16079334949113896, 0.2886103861680662, 0.16...","[0.01712255977963738, 0.027090324349444663, 0...."
2,2,tv_nuts,0.104234,0.699103,"[0.7535253763198853, 0.2065604329109192, 0.475...","[2.7720702342362147, 0.06997015001664085, 4.87...","[4.320941180856281, 0.8301907628735165, 10.063...","[0.08797597296601845, 0.05125281931954234, -0....","[0.2337470445744714, 0.0, 0.0, 0.0717390315349...","[0.10741934675276345, 0.0, 0.0, 0.0, 0.0860611..."
3,3,tv_nuts,0.341557,2.344679,"[0.36873242259025574, 0.5767415761947632, 0.76...","[10.49599799044622, 0.21310163093369586, 0.633...","[13.379044282151, 1.8779417580932, 1.326859943...","[0.010172471934946137, 0.06570099614669675, -0...","[0.3029368986249495, 0.11796947866432377, 0.10...","[0.0, 0.0, 0.0, 0.0, 0.021475551900879183, 0.0..."
4,4,tv_nuts,0.149083,0.142283,"[0.7817994356155396, 0.6081915497779846, 0.560...","[12.129046556640644, 0.19956457988281284, 0.28...","[26.220762918799107, 2.22348773149627, 1.34523...","[-0.10773505067690592, 0.11062700294980607, -0...","[0.0, 0.0, 0.0, 0.0, 0.44343777398069445, 0.0,...","[0.0, 0.04471783854829004, 0.09672474798116945..."
5,5,tv_nuts,0.377748,1.072810,"[0.2543151378631592, 0.25819700956344604, 0.71...","[0.5572417617404685, 0.5349406529763575, 0.160...","[1.6802851360243956, 1.7017982070956756, 1.956...","[0.06014740850041167, -0.009809137056882328, -...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7414027581476...","[0.0, 0.0, 0.0, 0.0, 0.01862499905216813, 0.0,..."
6,6,tv_nuts,0.160157,0.267231,"[0.2527131736278534, 0.29615518450737, 0.61198...","[0.11099265906353192, 0.7077022147963873, 4.57...","[0.6156601118974282, 1.9765241773540496, 4.924...","[0.14770123384788983, -0.13252530792165526, -0...","[0.0139217982242804, 0.49473053169555464, 0.42...","[0.2727870001452052, 0.24917411068031897, 0.06..."
7,7,tv_nuts,0.340967,0.312053,"[0.7179519534111023, 0.2091435194015503, 0.486...","[2.040634096878339, 0.07900162235086992, 1.215...","[2.0353214334784977, 0.8520630460231245, 1.166...","[0.018293359453413593, 0.13976017483644845, 0....","[0.0, 0.053757230752331506, 0.2623099684640309...","[0.0, 0.0, 0.0, 0.0, 0.0]"
8,8,tv_nuts,0.247196,0.095337,"[0.4183962047100067, 0.6244600415229797, 0.743...","[0.9321868626501912, 0.06538197676059651, 10.3...","[3.7614414265147955, 0.8683454461544444, 14.66...","[-0.03457852570527911, 0.11849542409193844, -0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.43516778833424963,...","[0.0, 0.107140500710213, 0.1198429332322585, 0..."
9,9,tv_nuts,0.809067,3.515835,"[0.32984814047813416, 0.2837252616882324, 0.58...","[1.839088581635389, 1.113206396367286, 0.19742...","[2.922496062970062, 3.996947056017796, 0.79562...","[0.09364271919003933, -0.0663289814390201, 0.0...","[0.12341152429889922, 0.8464139728055509, 0.0,...","[0.0, 0.0028081796163179256, 0.025831040323451..."


## Computational diagnostics ≠ point recovery ≠ uncertainty

Invalid MCMC exploration is unresolved, *not* a point failure. Short-smoke ADVI finite optimization only is **not** a convergence certificate; R-hat/ESS do not apply to variational draws. OLS has no posterior uncertainty. Pointwise credible intervals describe saved posterior uncertainty, not causal or frequentist generalization to new datasets.

In [4]:
diagnostics = pd.json_normalize([{'world': r['world_id'], 'method': r.get('method', evidence['kind']), 'status': r['computational_status'], **r['diagnostics']} for r in evidence['worlds']])
display(diagnostics.head(60))
recovery = pd.DataFrame([{'world': r['world_id'], 'method': r.get('method', evidence['kind']), 'status': r['computational_status'], 'media': r['recovery']['media_pass'], 'controls': r['recovery']['control_pass'], 'joint': r['recovery']['joint_pass'], 'total_media': (r['recovery']['total_media'] or {}).get('pass'), 'baseline': (r['recovery']['baseline'] or {}).get('pass'), 'uncertainty': r['uncertainty']['type']} for r in evidence['worlds']])
display(recovery.head(60))

,world,method,status,algorithm,diagnostics_pass,divergences,maxdepth_hits,max_rhat,min_bulk_ess,min_tail_ess,min_bfmi
0,0,tv_nuts,valid,NUTS,True,0,0,1.004876,2450.833308,1934.384524,0.829774
1,1,tv_nuts,valid,NUTS,True,0,0,1.002619,2259.360048,1530.912246,0.731343
2,2,tv_nuts,valid,NUTS,True,0,0,1.003166,2482.512946,1631.934337,0.723462
3,3,tv_nuts,valid,NUTS,True,0,0,1.003537,2890.146053,1698.576590,0.728383
4,4,tv_nuts,valid,NUTS,True,0,0,1.005039,2365.949698,1696.173889,0.776255
5,5,tv_nuts,valid,NUTS,True,0,0,1.002313,2780.994201,1957.231215,0.725424
6,6,tv_nuts,valid,NUTS,True,0,0,1.003701,1556.289249,1360.965717,0.720025
7,7,tv_nuts,valid,NUTS,True,0,0,1.002975,1880.115239,1601.005654,0.757505
8,8,tv_nuts,valid,NUTS,True,0,0,1.002365,1386.442228,1256.949335,0.762759
9,9,tv_nuts,valid,NUTS,True,0,0,1.004013,3058.362480,1706.576540,0.689438


,world,method,status,media,controls,joint,total_media,baseline,uncertainty
0,0,tv_nuts,valid,False,True,False,False,False,95% pointwise equal-tail posterior intervals
1,1,tv_nuts,valid,False,True,False,False,True,95% pointwise equal-tail posterior intervals
2,2,tv_nuts,valid,False,True,False,False,False,95% pointwise equal-tail posterior intervals
3,3,tv_nuts,valid,False,True,False,True,True,95% pointwise equal-tail posterior intervals
4,4,tv_nuts,valid,False,True,False,False,True,95% pointwise equal-tail posterior intervals
5,5,tv_nuts,valid,False,False,False,True,True,95% pointwise equal-tail posterior intervals
6,6,tv_nuts,valid,False,True,False,False,False,95% pointwise equal-tail posterior intervals
7,7,tv_nuts,valid,False,True,False,True,True,95% pointwise equal-tail posterior intervals
8,8,tv_nuts,valid,True,True,True,True,True,95% pointwise equal-tail posterior intervals
9,9,tv_nuts,valid,False,False,False,True,False,95% pointwise equal-tail posterior intervals


## Pattern-specific **channel** failure rates

Rows count **the channels bearing the named pattern**, not all failures in the same world. Patterns overlap and upward/downward trends have distinct denominators. Total includes known-zero channels; eligible excludes known-zero; unresolved eligible channels do not enter the failure-rate denominator.

In [5]:
for method, rates in evidence['pattern_rates'].items():
    display(Markdown('### ' + method))
    display(pd.DataFrame(rates)[['kind','pattern','total','eligible','known_zero','evaluable','failed','unresolved','failure_rate','status','small_count_warning']])
channel_table = pd.DataFrame(evidence['channels'])
display(channel_table.head(25))

### tv_nuts

,kind,pattern,total,eligible,known_zero,evaluable,failed,unresolved,failure_rate,status,small_count_warning
0,media,hf,441,214,227,214,130,0,0.607477,evaluable,False
1,media,pulse,340,156,184,156,93,0,0.596154,evaluable,False
2,media,onset,80,37,43,37,18,0,0.486486,evaluable,False
3,media,offset,39,17,22,17,11,0,0.647059,evaluable,False
4,media,flighting,93,45,48,45,14,0,0.311111,evaluable,False
5,media,level_jump,76,34,42,34,19,0,0.558824,evaluable,False
6,media,seasonal,137,73,64,73,42,0,0.575342,evaluable,False
7,media,trend,146,67,79,67,38,0,0.567164,evaluable,False
8,media,trend_up,72,35,37,35,19,0,0.542857,evaluable,False
9,media,trend_down,74,32,42,32,19,0,0.593750,evaluable,False


,world_id,kind,channel,patterns,nonzero_effect,computational_status,point_pass,point_error,point_allowance
0,0,media,C1,"[hf, pulse, offset, flighting, trend, trend_up]",False,valid,None,0.000000,0.886613
1,0,media,C2,"[hf, pulse, onset]",True,valid,False,3.218380,1.705871
2,0,media,C3,"[hf, pulse, onset, level_jump, trend, trend_up]",True,valid,True,1.377139,2.814847
3,0,media,C4,"[hf, onset, offset, seasonal]",False,valid,None,0.000000,0.886613
4,0,media,C5,"[hf, offset, level_jump, seasonal]",True,valid,False,1.139965,1.073390
5,0,control,Z1,"[hf, flighting, level_jump]",False,valid,None,0.000000,0.886613
6,0,control,Z2,"[hf, pulse, level_jump, seasonal]",False,valid,None,0.000000,0.886613
7,0,control,Z3,"[hf, offset, level_jump, seasonal]",True,valid,True,0.249355,0.886613
8,1,media,C1,"[hf, flighting, seasonal]",True,valid,True,0.415314,1.170374
9,1,media,C2,"[hf, trend, trend_down]",True,valid,False,1.852558,1.170374


## Selectable true/fitted trajectories

Select a world/method and a **nonzero** contribution. Separate Plotly panels show its contribution (and available 95% interval), raw media spend for media channels, and the baseline (always retained). Legends sit below the panels. Known-zero padded/null channels are hidden; absence is not a success.

In [6]:
keys = sorted(evidence['trajectories'])
selection = widgets.Dropdown(options=keys, value=keys[0], description='World · method:', layout=widgets.Layout(width='450px'))
def _eligible(key):
    if key not in evidence['trajectories']:
        return ('Baseline',)
    t = evidence['trajectories'][key]
    truth = np.asarray(t['truth'])
    return tuple(name for i, name in enumerate(t['terms']) if i == 0 or np.linalg.norm(truth[:, i]) > 0)
initial = _eligible(selection.value)
channel = widgets.Dropdown(options=initial, value=initial[0], description='Channel:', layout=widgets.Layout(width='300px'))
def _update_channels(change=None):
    valid = _eligible(selection.value)
    wanted = channel.value if channel.value in valid else valid[0]
    channel.options = valid
    channel.value = wanted
selection.observe(_update_channels, names='value')
def _view(key, name):
    if key not in evidence['trajectories'] or name not in _eligible(key):
        return None  # transient selector update, not a report/plot error
    t = evidence['trajectories'][key]
    ix = t['terms'].index(name)
    week = t['weeks']
    truth, fit = np.asarray(t['truth']), np.asarray(t['fitted'])
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=(name + ' contribution', 'Raw media spend (if media)', 'Baseline contribution'))
    def line(row, values, label, dash='solid'):
        fig.add_trace(go.Scatter(x=week, y=values, name=label, mode='lines', line={'dash': dash}, legendgroup=label, showlegend=True), row=row, col=1)
    line(1, truth[:, ix], 'True ' + name)
    line(1, fit[:, ix], 'Fitted ' + name, 'dash')
    if t['lower95'] is not None:
        low, high = np.asarray(t['lower95'])[:, ix], np.asarray(t['upper95'])[:, ix]
        fig.add_trace(go.Scatter(x=week, y=low, line={'width':0}, showlegend=False, hoverinfo='skip'), row=1, col=1)
        fig.add_trace(go.Scatter(x=week, y=high, fill='tonexty', fillcolor='rgba(80,80,180,.17)', line={'width':0}, name='95% pointwise interval'), row=1, col=1)
    if name.startswith('C'):
        line(2, np.asarray(t['raw_spend'])[:, int(name[1:])-1], 'Raw spend ' + name)
    line(3, truth[:, 0], 'True baseline')
    line(3, fit[:, 0], 'Fitted baseline', 'dash')
    status = next((r['computational_status'] for r in evidence['worlds'] if str(r['world_id']) + (':' + r['method'] if 'method' in r else '') == key), 'unknown')
    fig.update_layout(height=800, template='plotly_white', legend={'orientation':'h', 'y':-.13, 'x':0}, title=key + ' · ' + name + ' · ' + status)
    display(fig)
    return fig
display(widgets.HBox([selection, channel]))
interactive = widgets.interactive_output(_view, {'key': selection, 'name': channel})
display(interactive)

Output()

## Interpretation · run-bound review

Pending until an explicit reviewer approves this exact run into a separate sealed sibling review artifact. A present but invalid/cross-run review FAILS CLOSED. Optional held-out-world logistic screening, t-SNE and correlated-pair analysis are **deferred/unsupported**.

In [7]:
from studies.recovery.review_interpretation import attached_review
review = attached_review(Path.cwd())
if review is None:
    display(Markdown(Path('interpretation.md').read_text()))
else:
    display(Markdown('**Reviewed for run** `' + review['report_run_id'] + '` by **' + review['reviewer'] + '** · decision: approved · review `' + review['review_id'] + '`'))
    display(Markdown(review['interpretation']))

**Reviewed for run** `f2dccfb682b95c9e8da4ffdc0d3f8c5f4134098468d54ef4f8c8e974cfbbffb3` by **Teemu Säilynoja** · decision: approved · review `15a972d9798d4f1c7631264c77fc7fa4da317a6fb9d8658ff52db8334e405cea`

# Interpretation: media attribution with a varying baseline

**Report:** `studies/recovery/report-artifacts/tv50-profile-v2/`

**Exact report run ID:** `f2dccfb682b95c9e8da4ffdc0d3f8c5f4134098468d54ef4f8c8e974cfbbffb3`

This narrative describes the existing 50-world TV-MCMC study, not the four-world software smoke test. Approval, if granted, applies only to this report identity; it is recorded separately from this text.

## Main finding

**Healthy MCMC did not usually recover every individual media contribution, although total media contribution was recovered much more often.** Observation noise was small relative to aggregate sales, but often comparable to the temporal variation of a single media contribution. The results therefore support using this generator configuration as a challenging attribution benchmark—not treating its attribution targets as uniformly easy because aggregate noise is low.

This is a statement about recovery under this experiment's information, priors, sample length and error allowance. It is not proof of structural non-identifiability, a general limitation of Bayesian inference, or validation that the synthetic inputs resemble real spending.

## What was evaluated

The study contains 50 synthetic worlds, 104 reported weeks and 97 scored weeks (7–103). Earlier inputs supply carryover history. There are 236 nonzero media effects and 148 nonzero control effects. Media inputs combine a smoothed random walk with optional high-frequency variation, pulses, activity gates, level changes, seasonality and trends. Contributions use geometric adstock followed by Michaelis–Menten saturation. The intrinsic baseline varies over time; observation noise is a separate Gaussian component.

This is an **oracle-information benchmark**: fitting knows the zero/nonzero effect graph and generator reference anchors/supports. It does not infer the graph from data, and realized coefficients or baseline paths are not supplied as initialization. Performance should not be advertised as expected performance on an unprivileged real-world modeling task.

A channel passes when the L2 error of its estimated contribution trajectory is no larger than the greater of 10% of that channel's true L2 norm and 1% of the noiseless outcome's L2 norm. Thus this is not a pure 10% relative-error criterion; small effects can be judged against the whole-outcome floor. Joint recovery requires every nonzero media/control effect to pass, excluding the baseline. The total-media sum and baseline path are separate estimands with their own gates.

## Results

All 50 fits passed the prespecified MCMC diagnostic screen: no divergences or maximum-depth hits, maximum recorded R-hat 1.0069, minimum bulk ESS 1,062, minimum tail ESS 1,168 and minimum BFMI 0.624. That makes their recovery scores evaluable; it does not guarantee exploration of every posterior mode or accurate attribution.

| Estimand | Point-recovery passes |
|---|---:|
| Every individual media contribution in a world | 11/50 (22%) |
| Every individual media and control contribution | 9/50 (18%) |
| Total media contribution trajectory | 36/50 (72%) |
| Baseline contribution trajectory | 36/50 (72%) |

At channel level, **144/236 media contributions failed (61.0%)**, compared with **19/148 controls (12.8%)**. Channels from the same world are dependent; these are descriptive pooled counts, not 384 independent experiments. Larger worlds receive more weight in pooled channel summaries, and an all-channel world gate becomes harder to satisfy as the number of effects increases.

The better total-media point recovery is consistent with uncertainty in allocating effects across components rather than uniformly poor estimation of the aggregate media contribution. It does **not** establish narrow posterior uncertainty for that total, nor demonstrate that every individual failure is merely a compensating allocation error. The report's 95% intervals are pointwise intervals, not simultaneous trajectory guarantees or held-out predictive calibration.

## What distinguishes passing and failing media channels?

These descriptors use the **true generated trajectories and parameters**, not estimates from the failed fits. CV is temporal standard deviation divided by the absolute temporal mean; saturation is the realized unit response after adstock, averaged over the scored window.

| Median true descriptor | Passing channels (92) | Failing channels (144) |
|---|---:|---:|
| Mean saturation | 41.7% | 51.3% |
| Adstock decay alpha | 0.451 | 0.547 |
| Asymptotic beta | 2.41 | 2.63 |
| Raw-spend CV | 71.6% | 58.2% |
| Contribution CV | 34.1% | 22.9% |
| Contribution temporal SD / observation-noise SD | 1.57 | 0.78 |

Observation-noise SD divided by mean observed sales has a median of **1.86%**, with a middle 50% of **1.09–3.48%**. This aggregate ratio alone is a poor difficulty indicator: an individual contribution's variation can be smaller than the noise even when total sales are measured relatively cleanly.

The clearest descriptive contrast in this table is the variation remaining in the contribution relative to noise. Failing channels also tend to have less spend variation, higher adstock decay and greater saturation. The two groups' median beta values are close; this does not rule out differences elsewhere in their distributions. These features are related to each other and to world-level conditions; the table does not identify their independent or causal effects. Changing one parameter does not guarantee improved recovery.

## Pattern-specific failures

Each numerator below counts failures of **the channel carrying the pattern**, not failures elsewhere in a world containing that pattern. Only nonzero effects are eligible; all 50 MCMC worlds are evaluable here.

| Assigned media pattern | Failed / eligible | Failure rate |
|---|---:|---:|
| Upward trend | 19/35 | 54% |
| Downward trend | 19/32 | 59% |
| Seasonal | 42/73 | 58% |
| On/off flighting | 14/45 | 31% |
| Delayed start | 18/37 | 49% |
| Early end | 11/17 | 65% |
| Level jump | 19/34 | 56% |
| High-frequency component | 130/214 | 61% |
| Pulses | 93/156 | 60% |

Patterns overlap, so rows are not additive or mutually exclusive comparisons. Assignment does not imply that the component dominates the final observed input. Flighting has the lowest observed failure fraction in this sample, but this is not a controlled test of flighting: amplitude, adstock, saturation, other assigned patterns and world context differ too. The early-end group is small. In particular, **upward-trending media are not uniformly unrecoverable**.

## Implications for generator development

1. Assess generated tasks using transformed contribution variation relative to noise, not only raw-spend CV or noise as a percentage of total sales.
2. Preserve separate channel-level and aggregate estimands: a useful total-media estimate is not a substitute for accurate channel attribution.
3. Evaluate proposed generator changes with matched experiments and fresh recovery results. Descriptive associations do not justify automatically increasing variability or replacing innovation distributions.
4. Retain hard cases and report their prevalence unless the intended task population explicitly excludes them. A difficult synthetic case is not, by itself, unrealistic.

## Limits and provenance

This is one exploratory 50-world configuration, not a calibrated distribution of real datasets. There is no real-data reference here establishing that the noise, schedules, smoothing or response shapes are empirically representative. The signed baseline has no nonnegative floor; its interpretation should follow that generator definition rather than assume a strictly positive autonomous sales component.

Do not directly compare these rates with the constant-baseline bounded-OLS report as an algorithm ranking: the outcome/baseline regime and estimation objective differ. The short four-world smoke fits validate software execution only and are not additional evidence for these scientific conclusions.

The saved study has **legacy manifest/input-hash compatibility only**; original per-output and historical source hashes are unavailable. The new report binds the evidence it reads, but cannot retroactively certify the old fitting process. Conclusions must be reconsidered after a dataset/configuration change rather than copied to a new run.
